In [0]:
# Load a dataset config from the Git folder.
# The notebook runs from datasentry_v2/notebooks, so ../config is the config folder.

import os, json, shutil, yaml
from datetime import datetime, timezone

DATASET = "banking"
CONFIG_PATH = os.path.abspath(f"../config/{DATASET}.yaml")

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

env = cfg["environment"]
print(f"Loaded  : {CONFIG_PATH}")
print(f"Dataset : {cfg['dataset']} (version {cfg['version']})")
print(f"Tables  : {[t['name'] for t in cfg['tables']]}")

In [0]:
# Catch config mistakes BEFORE any data is processed.
# A typo in YAML should stop the pipeline with a clear message, not silently skip a rule.

KNOWN_CHECKS = {
    "not_null":    ["column"],
    "regex":       ["column", "pattern"],
    "in_set":      ["column", "values"],
    "castable":    ["column"],
    "not_future":  ["column"],
    "range":       ["column"],
    "unique":      ["column"],
    "foreign_key": ["column", "ref_table", "ref_column"],
    "sql":         ["expression"],
}
NEEDS_TYPED_COLUMN = {"castable", "not_future", "range"}
VALID_ACTIONS = {"AUTO_FIX", "QUARANTINE", "ESCALATE"}
VALID_FIXES = {"deduplicate", "map_values"}
TOP_LEVEL = ["dataset", "version", "environment", "source", "scoring", "tables"]


def validate_config(cfg):
    """Return a list of problems. An empty list means the config is valid."""
    errors = []
    for key in TOP_LEVEL:
        if key not in cfg:
            errors.append(f"missing top-level key '{key}'")
    if errors:
        return errors

    dimensions = set(cfg["scoring"]["severity"])
    seen_tables = []
    for t in cfg["tables"]:
        tname = t.get("name", "?")
        for key in ["name", "file_pattern", "primary_key", "rules"]:
            if key not in t:
                errors.append(f"{tname}: missing '{key}'")
        typed = t.get("columns", {}) or {}
        names = set()
        for r in t.get("rules", []):
            rname = r.get("name", "?")
            where = f"{tname}.{rname}"
            if rname in names:
                errors.append(f"{where}: duplicate rule name")
            names.add(rname)
            check = r.get("check")
            if check not in KNOWN_CHECKS:
                errors.append(f"{where}: unknown check '{check}'")
                continue
            for key in KNOWN_CHECKS[check]:
                if key not in r:
                    errors.append(f"{where}: '{check}' needs '{key}'")
            if r.get("dimension") not in dimensions:
                errors.append(f"{where}: unknown dimension '{r.get('dimension')}'")
            if r.get("action") not in VALID_ACTIONS:
                errors.append(f"{where}: unknown action '{r.get('action')}'")
            if check in NEEDS_TYPED_COLUMN and r.get("column") not in typed:
                errors.append(f"{where}: column '{r.get('column')}' needs a typed entry under 'columns'")
            if check == "range" and not any(k in r for k in ["gt", "gte", "lt", "lte"]):
                errors.append(f"{where}: range needs one of gt, gte, lt, lte")
            if check == "foreign_key" and r.get("ref_table") not in seen_tables:
                errors.append(f"{where}: ref_table '{r.get('ref_table')}' must be listed before '{tname}'")
            fix = r.get("fix")
            if r.get("action") == "AUTO_FIX" and not fix:
                errors.append(f"{where}: AUTO_FIX rule needs a 'fix'")
            if fix and r.get("action") != "AUTO_FIX":
                errors.append(f"{where}: 'fix' is only allowed on AUTO_FIX rules")
            if fix and fix.get("type") not in VALID_FIXES:
                errors.append(f"{where}: unknown fix type '{fix.get('type')}'")
        seen_tables.append(tname)
    return errors


problems = validate_config(cfg)
if problems:
    raise ValueError("Config is invalid:\n  " + "\n  ".join(problems))
print("Config is valid.")

In [0]:
from collections import Counter

dims, actions = Counter(), Counter()
print(f"{'TABLE':<14} {'RULES':>5}")
for t in cfg["tables"]:
    print(f"{t['name']:<14} {len(t['rules']):>5}")
    for r in t["rules"]:
        dims[r["dimension"]] += 1
        actions[r["action"]] += 1

total = sum(dims.values())
distinct = len({r["name"] for t in cfg["tables"] for r in t["rules"]})
print(f"\nTotal checks: {total}   Distinct rule names: {distinct}")
print("By dimension:", dict(dims))
print("By action   :", dict(actions))

In [0]:
# One row per rule, so analysts and Power BI can see exactly what is checked.
# replaceWhere rewrites only this dataset's rows, leaving other datasets untouched.

LOADED_AT = datetime.now(timezone.utc)
severity = cfg["scoring"]["severity"]
CORE = {"name", "check", "column", "dimension", "action", "fix"}

rows = []
for t in cfg["tables"]:
    for i, r in enumerate(t["rules"], start=1):
        params = {k: v for k, v in r.items() if k not in CORE}
        rows.append((
            cfg["dataset"], t["name"], i, r["name"], r["check"], r.get("column"),
            r["dimension"], severity[r["dimension"]], r["action"],
            (r.get("fix") or {}).get("type"), json.dumps(params), cfg["version"], LOADED_AT,
        ))

schema = """dataset STRING, table_name STRING, rule_order INT, rule_name STRING,
            check_type STRING, column_name STRING, dimension STRING, severity STRING,
            action STRING, fix_type STRING, params STRING, config_version INT,
            loaded_at TIMESTAMP"""
registry = spark.createDataFrame(rows, schema)

target = f"{env['catalog']}.{env['schemas']['config']}.rule_registry"
if spark.catalog.tableExists(target):
    (registry.write.mode("overwrite")
        .option("replaceWhere", f"dataset = '{cfg['dataset']}'")
        .saveAsTable(target))
else:
    registry.write.mode("overwrite").saveAsTable(target)

print(f"{registry.count()} rules written to {target}")

In [0]:
# Keep an exact copy of the config, timestamped, in the governed Volume.

snap_dir = env["paths"]["config_snapshots"]
os.makedirs(snap_dir, exist_ok=True)
snap_file = f"{snap_dir}/{cfg['dataset']}_v{cfg['version']}_{LOADED_AT:%Y%m%d_%H%M%S}.yaml"
shutil.copy(CONFIG_PATH, snap_file)
print("Snapshot saved:", snap_file)

In [0]:
%sql
SELECT table_name, rule_order, rule_name, check_type, column_name, dimension, severity, action, fix_type
FROM workspace.dq_config.rule_registry
WHERE dataset = 'banking'
ORDER BY table_name, rule_order